In [1]:
import os
import time
import logging 
import pandas as pd
import re
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from io import StringIO

In [ ]:
# Configuração do caminho do WebDriver
EDGE_DRIVER_PATH = r"C:\Users\luiz.smorais\Documents\MINISTERIO DA SAUDE\COMPRASLOG\DE-PARA SEI COMPRAS\web-driver\msedgedriver.exe"

download_dir = r"C:\Users\luiz.smorais\Documents\MINISTERIO DA SAUDE\COMPRASLOG\DE-PARA SEI COMPRAS\downloads"

# Configuração de logs
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Configuração do WebDriver para o Edge
try:
    edge_options = Options()
    
    # Configuração do diretório de downloads
    edge_options.add_experimental_option('prefs', {
        'download.default_directory': download_dir,
        'download.prompt_for_download': False,
        'download.directory_upgrade': True,
        'safebrowsing.enabled': True
    })
    
    # Inicializando o WebDriver do Edge apontando para o driver manualmente
    service = Service(EDGE_DRIVER_PATH)
    driver = webdriver.Edge(service=service, options=edge_options)
    logging.info("WebDriver do Edge inicializado com sucesso.")
except Exception as e:
    logging.error(f"Erro ao configurar o WebDriver: {str(e)}")
    exit()

# Função para substituir caracteres especiais por _
def substituir_caracteres_especiais(nome):
    return re.sub(r'[^\w\s]', '_', nome)

# Garantir que o diretório de download exista
try:
    if not os.path.exists(download_dir):
        os.makedirs(download_dir)
        logging.info(f"Diretório de download criado: {download_dir}")
except Exception as e:
    logging.error(f"Erro ao criar o diretório de download: {str(e)}")
    driver.quit()
    exit()

# Acessar o site e realizar login
driver.get('https://sei.saude.gov.br/')
logging.info("Página de login do SEI carregada com sucesso.")

# Obter valores das variáveis de ambiente
usuario_env = os.getenv('USUARIO')
senha_env = os.getenv('SENHA')
orgao_env = os.getenv('ORGAO')

# Preencher os campos de login com as variáveis carregadas
try:
    usuario = WebDriverWait(driver, 60).until(EC.presence_of_element_located((By.ID, 'txtUsuario')))
    senha = WebDriverWait(driver, 60).until(EC.presence_of_element_located((By.ID, 'pwdSenha')))
    orgao = WebDriverWait(driver, 60).until(EC.presence_of_element_located((By.ID, 'selOrgao')))

    usuario.send_keys(usuario_env)
    senha.send_keys(senha_env)
    orgao.send_keys(orgao_env)
    senha.send_keys(Keys.RETURN)
    logging.info("Login realizado com sucesso.")
except Exception as e:
    logging.error(f"Erro durante o login: {str(e)}")
    driver.quit()
    exit()

# Carregar os números de processo a partir do arquivo CSV
try:
    csv_path = r'C:\Users\luiz.smorais\Documents\MINISTERIO DA SAUDE\COMPRASLOG\DE-PARA SEI COMPRAS\processos.csv'
    df_documentos = pd.read_csv(csv_path, sep=';', dtype={'Processo': str}, encoding='latin1')
    logging.info("Arquivo CSV carregado com sucesso.")
except Exception as e:
    logging.error(f"Erro ao carregar o arquivo CSV: {str(e)}")
    driver.quit()
    exit()

# Iterar pelos processos no CSV
for index, row in df_documentos.iterrows():
    Processo = str(row['Processo'])
    nome_arquivo = substituir_caracteres_especiais(Processo)
    logging.info(f"Iniciando a busca do processo {Processo}.")
    
    try:
        pesquisa = WebDriverWait(driver, 60).until(
            EC.presence_of_element_located((By.ID, 'txtPesquisaRapida'))
        )
        pesquisa.clear()
        pesquisa.send_keys(Processo)
        pesquisa.send_keys(Keys.RETURN)
        logging.info(f"Processo {Processo} pesquisado.")
        time.sleep(2)
    except Exception as e:
        logging.error(f"Erro ao pesquisar o processo {Processo}: {str(e)}")

# Fechar o navegador
driver.quit()
logging.info("Navegador fechado.")


2025-02-28 12:03:51,538 - INFO - WebDriver do Edge inicializado com sucesso.
2025-02-28 12:03:55,193 - INFO - Página de login do SEI carregada com sucesso.
2025-02-28 12:03:58,285 - ERROR - Erro durante o login: 'NoneType' object is not iterable
2025-02-28 12:04:01,052 - INFO - Arquivo CSV carregado com sucesso.
2025-02-28 12:04:01,098 - INFO - Iniciando a busca do processo 25000.037991/2024-92.
2025-02-28 12:04:05,176 - WARNING - Retrying (Retry(total=2, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000013CBDA9C690>: Failed to establish a new connection: [WinError 10061] Nenhuma conexão pôde ser feita porque a máquina de destino as recusou ativamente')': /session/7cd2a2ecf97488b20ff1714740faa5bd/element
2025-02-28 12:04:09,238 - WARNING - Retrying (Retry(total=1, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NewConnectionError('<urllib3.connection